In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import IsolationForest

In [ ]:
conn = sqlite3.connect("../database/air_quality.db")

df = pd.read_sql_query(
    "SELECT * FROM air_quality",
    conn
)

conn.close()

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day

df = df.dropna(subset=["calculated_aqi"])

print(f"Loaded {len(df)} rows")
df.head()

## Isolation Forest — EPA AQI (0–500)

Detects anomalous air-quality readings using the EPA AQI (0–500) and
pollutant concentrations. Approximately 5% of readings are expected to
be anomalous (`contamination=0.05`).

In [ ]:
features = [
    "calculated_aqi",
    "pm2_5",
    "pm10",
    "co",
    "no2",
    "o3",
    "so2"
]

X = df[features]

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X)

In [ ]:
df["anomaly"] = model.predict(X)

# 1 = normal, -1 = anomaly
counts = df["anomaly"].value_counts()
n_normal = counts.get(1, 0)
n_anomaly = counts.get(-1, 0)

print(f"Normal readings:  {n_normal}")
print(f"Anomalous readings: {n_anomaly}")
print(f"Anomaly rate: {n_anomaly / len(df) * 100:.1f}%")

In [ ]:
joblib.dump(model, "saved_models/isolation_forest_aqi_500.pkl")

loaded_model = joblib.load("saved_models/isolation_forest_aqi_500.pkl")
print(f"feature_names_in_: {list(loaded_model.feature_names_in_)}")

sample = X.iloc[[0]]
prediction = loaded_model.predict(sample)
print(f"Sample prediction: {prediction[0]} (1=normal, -1=anomaly)")

In [ ]:
anomalies = df[df["anomaly"] == -1]

plt.figure(figsize=(12, 6))
plt.plot(df.index, df["calculated_aqi"], label="AQI", alpha=0.8)
plt.scatter(
    anomalies.index,
    anomalies["calculated_aqi"],
    color="red",
    label="Anomaly",
    zorder=5
)
plt.title("EPA AQI Anomaly Detection (0–500)")
plt.xlabel("Index")
plt.ylabel("AQI")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()